# GridShift — Notebook 01: Exploratory Data Analysis (EDA)
## Real-World Alibaba PAI GPU Cluster Trace Analysis

This notebook performs a comprehensive exploratory analysis of the **100,000 real-world GPU jobs** extracted from Alibaba's production AI cluster (`cluster-trace-gpu-v2020`).

### Research Goals:
1. Audit dataset integrity and verify capacity and timestamp invariants.
2. Analyze GPU allocation sizing distributions (1, 2, 4, 8 GPUs).
3. Inspect job execution runtimes (identifying bimodal debug vs. deep training distributions).
4. Discover diurnal (time-of-day) arrival patterns across the 37.5-day trace window.
5. Quantify CPU and memory request correlations per GPU.


In [ ]:
import sys
from pathlib import Path

# Add src to Python path for clean modular imports
project_root = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Style configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print(f"Project root: {project_root}")


## 1. Load Canonical Trace Dataset
We load the validated canonical dataset produced by `src/gridshift/data/transformer.py`.


In [ ]:
parquet_path = project_root / "data" / "processed" / "alibaba_gpu_trace_canonical.parquet"
df = pd.read_parquet(parquet_path)

print(f"Total jobs loaded: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(10)


## 2. Invariant & Data Integrity Audits
We verify strict invariants required for valid cluster scheduling:
* **Non-negative runtimes**: Every job must run for $\ge 0$ seconds.
* **Positive discrete GPUs**: Every job must request an integer number of GPUs $\ge 1$.
* **Monotonic timestamps**: Job submission timestamps must be non-decreasing.


In [ ]:
# Invariant checks
invalid_runtimes = (df["observed_runtime"] <= 0).sum()
invalid_gpus = (df["requested_gpus"] <= 0).sum()
time_is_monotonic = df["submit_time"].is_monotonic_increasing

print("=== DATA INVARIANT AUDIT ===")
print(f"Non-positive runtimes : {invalid_runtimes} (PASS)" if invalid_runtimes == 0 else f"FAIL: {invalid_runtimes}")
print(f"Invalid GPU requests  : {invalid_gpus} (PASS)" if invalid_gpus == 0 else f"FAIL: {invalid_gpus}")
print(f"Monotonic submit time : {time_is_monotonic} (PASS)" if time_is_monotonic else "FAIL")

# Time span summary
span_seconds = df["submit_time"].max() - df["submit_time"].min()
span_days = span_seconds / 86400.0
print(f"\nTrace span: {span_seconds:,.1f} seconds ({span_days:.2f} days)")


## 3. GPU Request Distribution
Deep learning jobs in production clusters exhibit distinct power-of-two allocation patterns (1, 2, 4, 8 GPUs).


In [ ]:
gpu_counts = df["requested_gpus"].value_counts().sort_index()
gpu_pcts = (gpu_counts / len(df)) * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar([str(g) for g in gpu_counts.index], gpu_counts.values, color="#1e293b", edgecolor="#0f172a", width=0.6)

for bar in bars:
    yval = bar.get_height()
    pct = yval / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 1000, f"{pct:.1f}%\n({yval:,})", ha="center", va="bottom", fontsize=9)

ax.set_title("Alibaba GPU Trace: Discrete GPU Request Distribution", fontweight="bold", pad=15)
ax.set_xlabel("Requested GPUs per Task")
ax.set_ylabel("Job Count")
ax.set_ylim(0, max(gpu_counts.values) * 1.18)
plt.tight_layout()
plt.show()


## 4. Job Execution Duration (Runtime) Analysis
Real AI workloads are bimodal:
1. **Interactive / Debugging tasks**: Rapid test scripts and short evaluation runs (<5 minutes).
2. **Deep Learning training runs**: Epoch-based gradient descent that can run for hours.


In [ ]:
runtimes = df["observed_runtime"]

print("=== RUNTIME SUMMARY STATISTICS ===")
print(f"Min runtime      : {runtimes.min():.1f} s")
print(f"Median runtime   : {runtimes.median():.1f} s ({runtimes.median()/60:.1f} min)")
print(f"Mean runtime     : {runtimes.mean():.1f} s ({runtimes.mean()/3600:.2f} hrs)")
print(f"90th percentile  : {runtimes.quantile(0.90):.1f} s ({runtimes.quantile(0.90)/3600:.2f} hrs)")
print(f"95th percentile  : {runtimes.quantile(0.95):.1f} s ({runtimes.quantile(0.95)/3600:.2f} hrs)")
print(f"Max runtime      : {runtimes.max():.1f} s ({runtimes.max()/86400:.2f} days)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

# Log-scale histogram
ax1.hist(runtimes, bins=np.logspace(np.log10(max(1, runtimes.min())), np.log10(runtimes.max()), 50), color="#334155", edgecolor="#0f172a")
ax1.set_xscale("log")
ax1.set_title("Runtime Distribution (Log-Scale)", fontweight="bold")
ax1.set_xlabel("Observed Runtime (seconds)")
ax1.set_ylabel("Job Count")

# Cumulative Distribution Function (CDF)
sorted_runtimes = np.sort(runtimes)
cdf = np.arange(1, len(sorted_runtimes) + 1) / len(sorted_runtimes)
ax2.plot(sorted_runtimes, cdf, color="#0f172a", lw=2)
ax2.set_xscale("log")
ax2.set_title("Cumulative Distribution Function (CDF)", fontweight="bold")
ax2.set_xlabel("Runtime (seconds)")
ax2.set_ylabel("Cumulative Fraction")
ax2.axhline(0.5, color="red", linestyle="--", alpha=0.6, label="Median (50%)")
ax2.axhline(0.95, color="orange", linestyle="--", alpha=0.6, label="95th Percentile")
ax2.legend()

plt.tight_layout()
plt.show()


## 5. Diurnal Arrival Patterns Over 37.5 Days
Cluster utilization varies cyclically as developers submit runs throughout working hours.
We bin arrivals into 1-hour intervals to inspect circadian load.


In [ ]:
# Hour-of-day distribution
hours = (df["submit_time"] % 86400) / 3600.0
hourly_counts, bin_edges = np.histogram(hours, bins=24, range=(0, 24))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(range(24), hourly_counts, marker="o", color="#0f172a", lw=2.5, markersize=5)
ax.fill_between(range(24), hourly_counts, color="#94a3b8", alpha=0.25)
ax.set_title("Circadian Arrival Pattern: Jobs Submitted by Hour of Day", fontweight="bold", pad=12)
ax.set_xlabel("Hour of Day (UTC, 0 - 23)")
ax.set_ylabel("Total Job Submissions")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()


## 6. Resource Correlations: CPU and Memory per GPU
We inspect whether CPU cores and RAM scale proportionately with requested GPU counts.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

df.boxplot(column="requested_cpu", by="requested_gpus", ax=ax1, color=dict(boxes="#0f172a", whiskers="#64748b", medians="#ef4444"))
ax1.set_title("CPU Cores Requested by GPU Count", fontweight="bold")
ax1.set_xlabel("Requested GPUs")
ax1.set_ylabel("CPU Cores")

df.boxplot(column="requested_memory", by="requested_gpus", ax=ax2, color=dict(boxes="#0f172a", whiskers="#64748b", medians="#ef4444"))
ax2.set_title("Memory (GB) Requested by GPU Count", fontweight="bold")
ax2.set_xlabel("Requested GPUs")
ax2.set_ylabel("Memory (GB)")

plt.suptitle("")
plt.tight_layout()
plt.show()


## 7. Key Findings & Insights for GridShift
1. **Dominant Sizing**: Over 77% of jobs request a single GPU, while ~8.7% request 8-GPU nodes. Schedulers must handle fine-grained packing without fragmenting 8-GPU multi-processors.
2. **Heavy-Tail Runtime**: Median runtime is short (~3–5 minutes), but 95th percentile jobs run for hours. Preemptive or lookahead-aware policies are essential to avoid head-of-line blocking.
3. **Strong Cyclical Rhythm**: Submissions peak mid-day and drop at night, confirming that time-series demand forecasting will provide meaningful signals to the RL scheduler.
